# Overlapping CCZ gates as a four-qubit phase polynomial

Derive an overlapping three-body phase oracle, verify its 16-by-16 operator, and connect the final amplitudes to a Walsh–Hadamard transform. This extends two-qubit controlled-phase reasoning to hypergraph states. You need complex vectors, binary arithmetic, and the Qiskit bit-ordering convention.

**Prerequisites:** Python 3.12 or later and the tested environment in the [collection README](../../../README.md). Install the [root requirements](../../../requirements.txt) into an isolated environment before opening this notebook; select that environment's kernel. Run all cells from a fresh kernel in order. This notebook makes no network calls, reads no external files, and writes no artifacts by default.

**Reproducibility:** all examples and data here are synthetic. Simulator and sampling seeds are explicit. Assertions distinguish mathematical identities, finite-shot checks, and integration contracts. Approximate comparisons state their tolerance.


Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. The Boolean polynomial controls the phase

Apply CCZ on $(q_0,q_1,q_2)$ and $(q_1,q_2,q_3)$. For a computational-basis string $x$, the phase is
$(-1)^{f(x)}$, with $f(x)=x_0x_1x_2\oplus x_1x_2x_3$. The two gates commute because both are diagonal. If both products are one, their two minus signs cancel: $|1111\rangle$ is unchanged. An overlapping CCZ construction is therefore not an OR of its conditions.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector


def show_plot():
    import io

    from IPython.display import Image, display

    buffer = io.BytesIO()
    plt.gcf().savefig(buffer, format="png", dpi=120, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
    plt.close()


n = 4
bits = np.array([[(basis >> q) & 1 for q in range(n)] for basis in range(2**n)])
f = (bits[:, 0] * bits[:, 1] * bits[:, 2] + bits[:, 1] * bits[:, 2] * bits[:, 3]) % 2
D = np.diag((-1.0) ** f)
phase = QuantumCircuit(n)
phase.ccz(0, 1, 2)
phase.ccz(1, 2, 3)
assert np.allclose(Operator(phase).data, D, atol=1e-12)
assert D[15, 15] == 1 and D[7, 7] == -1 and D[14, 14] == -1
reverse = QuantumCircuit(n)
reverse.ccz(1, 2, 3)
reverse.ccz(0, 1, 2)
assert np.allclose(Operator(reverse).data, D)
pd.DataFrame({"q3q2q1q0": [f"{i:04b}" for i in range(16)], "phase": np.diag(D)})

## 2. Compute the complete basis change

For $W=H^{\otimes4}$, $U=WDW$ is unitary, Hermitian, and an involution. Its action on $|0000\rangle$ has amplitudes
$a_y=2^{-4}\sum_x(-1)^{f(x)+x\cdot y}$.
This is the normalized Walsh spectrum of the phase function; squares of those amplitudes give measurement probabilities. Use an independent integer-parity calculation to check the circuit rather than comparing two calls to the same simulator.


In [ ]:
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
W = H
for _ in range(n - 1):
    W = np.kron(W, H)
U = W @ D @ W
circuit = QuantumCircuit(n)
circuit.h(range(n))
circuit.compose(phase, inplace=True)
circuit.h(range(n))
assert np.allclose(Operator(circuit).data, U, atol=1e-12)
assert np.allclose(U.conj().T @ U, np.eye(16), atol=1e-12)
assert np.allclose(U @ U, np.eye(16), atol=1e-12)
walsh = np.array(
    [sum((-1) ** (int(f[x]) + (x & y).bit_count()) for x in range(16)) / 16 for y in range(16)]
)
state = Statevector.from_instruction(circuit)
assert np.allclose(state.data, walsh, atol=1e-12)
assert np.isclose(np.sum(walsh**2), 1)
print(
    pd.DataFrame(
        {"basis": [f"{i:04b}" for i in range(16)], "amplitude": walsh, "probability": walsh**2}
    ).query("probability > 0")
)

## 3. Prove why pairwise CZ gates cannot replace CCZ

Any product of CZ gates contributes a degree-two polynomial over $\mathbb F_2$. Restrict the four-qubit example to $x_3=0$: the desired sign is $(-1)^{x_0x_1x_2}$. Its coefficient on the cubic monomial is one. Enumerating the eight choices of pairwise CZ gates on three qubits confirms that none matches that truth table. This is a limitation of this **diagonal, CZ-only** gate family, not of universal quantum circuits.

The left plot shows real operator entries in `q3q2q1q0` basis order; selected axis labels identify the discrete rows and columns. The right plot shows probabilities for one specific input. Matrix-entry colors are amplitudes, so they must not be interpreted as those probabilities.


In [ ]:
target = np.array([(-1) ** (((x >> 0) & 1) * ((x >> 1) & 1) * ((x >> 2) & 1)) for x in range(8)])
pairs = [(0, 1), (0, 2), (1, 2)]
for mask in range(8):
    candidate = np.array(
        [
            (-1)
            ** sum(
                ((x >> a) & 1) * ((x >> b) & 1) for k, (a, b) in enumerate(pairs) if (mask >> k) & 1
            )
            for x in range(8)
        ]
    )
    assert not np.array_equal(candidate, target)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
heatmap = axes[0].imshow(U.real, cmap="RdBu_r", vmin=-1, vmax=1)
basis_ticks = [0, 5, 10, 15]
basis_labels = [f"{index:04b}" for index in basis_ticks]
axes[0].set_xticks(basis_ticks, labels=basis_labels)
axes[0].set_yticks(basis_ticks, labels=basis_labels)
axes[0].set(
    xlabel="Input basis (q3q2q1q0)",
    ylabel="Output basis (q3q2q1q0)",
    title="Complete conjugated operator",
)
fig.colorbar(heatmap, ax=axes[0], label="Real matrix entry")
axes[1].bar(range(16), walsh**2)
axes[1].set_xticks([0, 3, 6, 9, 12, 15])
axes[1].set(xlabel="Basis index (integer)", ylabel="Probability", title="Output for |0000⟩")
show_plot()

## Limits, cleanup, and next steps

The explicit matrices and truth-table enumeration grow exponentially. They are useful verification tools for small circuits, not a scalable synthesis strategy. Try adding a third hyperedge, derive its Boolean phase, and check cancellations before interpreting a histogram.

No persistent artifacts or external services are used. Restart the kernel to release arrays and figures.

Primary references: [CCZGate API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.CCZGate), [Operator API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Operator).
